# Etterforskning

Vi har en stor ferdiglaget graf, med titusenvis av noder.  Vi skal bruke Neo4J til å lete etter kriminalitet.


In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

Om dette feiler, forsøk igjen om en liten stund; databasen skal laste ned utvidelser og initalisere seg.  Den feiler om den ikke har nett.

In [22]:
# Få Kontakt
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
# Ta et null-kall for å sjekke at det er liv - detaljer nedenfor
records, summary, keys = driver.execute_query(
    """RETURN null""")
print(f"Server: {summary.server.address}")

Server: 127.0.0.1:7687


Vi flytter grafen vi skal arbeide med

In [2]:
%%bash
cp grafer/Komplett.graphml neo4j/import

### Rydde og gjøre klart
Hente alle projeeksjoner og slette dem, og slette alle noder.

In [34]:
# For ordens skyld, i tilfelle databasen har blitt brukt, la oss slette
# alle GDS-grafer.  
records, summary, keys = driver.execute_query(
    """
    CALL gds.graph.list() YIELD graphName
    WITH graphName
    CALL gds.graph.drop(graphName) YIELD graphName AS borte
    RETURN borte
    """)
for s in summary.gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#

Hvordan gikk det: note: successful completion


In [35]:
# Fjerne eventuelle noder
records, summary, keys = driver.execute_query(
    """
    MATCH (N)
    DETACH DELETE N
    RETURN COUNT(N)
    """)
for s in summary.gql_status_objects:
    print(f"Hvordan gikk det: {s}")
#
for r in records:
    innhold = r.data()
    print(innhold)
#

Hvordan gikk det: note: successful completion
{'COUNT(N)': 56831}


### Grafen

Vi leser inn grafen vi har laget

In [36]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """
    CALL apoc.import.graphml(
        "Komplett.graphml", 
        {storeNodeIds: true, readLabels: true})
    """)
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tAlt klart: {summary.result_consumed_after} ms")

	file: Komplett.graphml
	source: file
	format: graphml
	nodes: 56831
	relationships: 103587
	properties: 57673
	time: 1138
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Alt klart: 1140 ms


## Lete etter et esel
Vi ser etter en klikk hvor alle kjenner hverandre.  Vi finner kanskje mange.  Det er interessant om (nesten) alle (eller i det minste flertallet) har uttak av kontanter.
Dette er typisk GDS-mat (algoritmer som kjører på hele grafen, ikke bare på enkeltnoder).

In [37]:
# Først, sjekke at det vi leter faktisk er der (sånn for sikkerhets skyld)
records, summary, keys = driver.execute_query(
  """
    MATCH (N:Person {Esel:True})
    RETURN count(N)
  """
)
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"{k}: {record_dict[k]}")
#

count(N): 16


Vi leter etter en klikk, hvor alle kjenner hverandre.  Vi starter med å hente ut alle personer.  Vi henter altså ikke ut konti og transkaskjoner.  Det er mer enn 100.000 kanter i EPOST-settet så mye blir ikke med inn.

In [38]:
# Hente personer og kun Kjenner-relasjonen
records, summary, keys = driver.execute_query(
  """CALL 
    gds.graph.project(
      'EselGraf',
      'Person',              // Type på nodene
      {
        Kjenner: {           //  Type på kanter
          type: 'Kjenner',
          orientation: 'UNDIRECTED' // Betrakter dem som om de var uten retning
        }
      }
    )
  YIELD nodeCount, relationshipCount, projectMillis;
  """
)
# Det skal ikke komme data tilbake men info om subgrafen.  Det vil si det kommer
# en dict med info om hvor mange noder og kanter som ble hentet
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

	nodeCount: 56674
	relationshipCount: 806
	projectMillis: 17
Ressursbruk
	Kjøringen: 1ms


Nå trenger vi en liste over folk i klikker, slik at vi vet hvor vi skal lete.  SOm vi så tidligere så er klikk en "sterk" datastruktur som ikke så lett oppstår tilfeldig, heller ikke i grafer uten skala.

I virkeligheten ville vi ha forsøkt en rekke verdier og undersøkt settet for hver verdi.  Vi kan imidlertid kortslutte litt

In [45]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('EselGraf')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS N, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    RETURN COUNT(N)
    """
)
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

<Record COUNT(N)=15>
Ressursbruk
	Kjøringen: 1ms
	Å konsumere: 53ms


Vi ser vi trolig har en klikk her.  La oss merke disse nodene, og deretter skrive dem tilbake til databasen slik at vi kan vise dem frem i nettleseren.

In [ ]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('EselGraf')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS N, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    SET N.EselMistanke = 1
    RETURN N
    """
)
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

<Record N=<Node element_id='4:757c1468-8c88-46ea-90c0-a42666a5f63f:56317' labels=frozenset({'Person'}) properties={'id': 'c99442ca-1b9f-4f27-b87c-2cc58ffe5262', 'Esel': True, 'EselMistanke': 1, 'Navn': 'Jon'}>>
<Record N=<Node element_id='4:757c1468-8c88-46ea-90c0-a42666a5f63f:56319' labels=frozenset({'Person'}) properties={'id': '6ece721f-be85-4ea5-9c23-911bc2c4c29a', 'Esel': True, 'EselMistanke': 1, 'Navn': 'Ada'}>>
<Record N=<Node element_id='4:757c1468-8c88-46ea-90c0-a42666a5f63f:56321' labels=frozenset({'Person'}) properties={'id': '1d4149a9-8a70-43d3-9766-24b4029f769e', 'Esel': True, 'EselMistanke': 1, 'Navn': 'Jasmin'}>>
<Record N=<Node element_id='4:757c1468-8c88-46ea-90c0-a42666a5f63f:56323' labels=frozenset({'Person'}) properties={'id': 'eee97ada-67ca-4c8d-9a3a-1c1e052969eb', 'Esel': True, 'EselMistanke': 1, 'Navn': 'Eldar'}>>
<Record N=<Node element_id='4:757c1468-8c88-46ea-90c0-a42666a5f63f:56325' labels=frozenset({'Person'}) properties={'id': '1318fd04-a6e0-4821-afd5-c973a

Vi har merket nodene i projeksjonen, skriv det vi har merket tilbake til databsen (slik at vi kan se på resultatet)

In [ ]:
XXX
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.write('EselGraf', {
        writeProperty: 'EselMistanke'
    })
    YIELD nodePropertiesWritten, degeneracy
    """
)
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

<Record nodePropertiesWritten=56674 degeneracy=14>
Ressursbruk
	Kjøringen: 1ms


Om alt fungerer som det skal, da vil denne koden i nettleseren finne Eselet for oss:
```
match (n:Person {EselMistanke:1}) return n limit 20
```

In [16]:
# Opprydding
# Fjerne projeksjonen
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('VårGraf', false) YIELD graphName"""
)
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (N:Person  {EselMistanke:1})
    REMOVE N.EselMistanke
    RETURN COUNT(N) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#


	Antall: 28


## Lete etter en bakmann

Vi leter etter et sett med klikker, som alle har egenskapen at de er knyttet sammen gjennom en Bakmann.  Det vil i praksis si en "stjerne" med bakmannen i "sentrum".

Vi gjør noen antagelser (som vi kan eksperimentere med senere):
- Gjengene er (ekte) klikker;
- Klikkene har minst fem medlemmer (med hundre tusen noder er det "uendelig" mange mindre klikker);
- En Bakmann har (minst) tre klikker han håndterer, og
- Bakmannen er ikke med i klikken, altså at bakmannen ikke "Kjenner" alle medlemmene.


Strategien blir:
1. Lage en projeksjon av alle personer
2. Finne *local cluster coefficient* (*LCC*) for alle noder (vi har hentet inn), og skriv verdien tilbake i databasen.  Vi gikk gjennom *LCC* i den generelle delen;
3. En bakmann vil ha (mange) færre venner (relasjonen Kjenner) enn medlemmer av klikkene han "håndterer".  Derfor henter vi ut Personer med få naboer (lav LCC);
4. Som Person har Bakmannen konto.  Siden vi har antatt at han vil "håndtere" (minst) tre klikker vil ha (minst) fire relasjoner, og
5. Bakmannens venner, derimot, er med i klikker og har høy LCC (minst 0.7)

Altså: etter å ha merket nodene med `LCC` finner vi noder med `LCC < 0.1` men som har naboer med `LCC > 0.7`.  Vi merker dem med `MISTENKT` for inspeksjkon.

In [17]:
# Fjerne projeksjonen om den finnes
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('BakmannGraf', false) YIELD graphName"""
)
# Fjerne merkingne
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#

In [18]:
# Lag projeksjonen
records, summary, keys = driver.execute_query(
"""
  CALL gds.graph.project(
  'BakmannGraf',
  'Person',
  {
    Kjenner: { orientation: 'UNDIRECTED' }  // Fjerne eventuell retning
  }
  )
  YIELD nodeCount, relationshipCount, projectMillis;
""")

for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodeCount: 56674
	relationshipCount: 806
	projectMillis: 25
Ressursbruk
	Kjøringen: 17ms
	Å konsumere: 28ms


In [19]:
# Regn ut LCC og skriv tilbake på hver node (som er hentet inn)
# Det betyr at noder av andre typer enn Person ikke får denne
records, summary, keys = driver.execute_query(
"""    
  CALL gds.localClusteringCoefficient.write('BakmannGraf', {
  writeProperty: 'LCC'
});
""")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

Ressursbruk
	Kjøringen: 22ms


In [20]:
# Kjøre punktene 3, 4, og 5 over
records, summary, keys = driver.execute_query(
"""
  MATCH (Mistenkt:Person)
  WHERE Mistenkt.LCC < 0.1  // Ikke alle noder har verdi
  AND count{(Mistenkt)--()} >=4 // Minst fire relasjoner
  MATCH (Mistenkt:Person)--(nabo:Person) // Finn naboene
  WHERE nabo.LCC > 0.7 // ...og bare naboer med mange venner
  WITH Mistenkt, collect(nabo) AS Kandidater
  WHERE size(Kandidater) >= 4   // og bare de som har minst tre klikker
  // Når vi kommer hit, da tror vi at vi har den mistenkte
  SET Mistenkt.MISTENKT = 1
  return Mistenkt
""")
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	Mistenkt: {'EselMistanke': 5, 'Bakmann': True, 'id': '01f59946-3b7e-455d-be70-1ecd9de9b32d', 'MISTENKT': 1, 'LCC': 0.0, 'Navn': 'Bakmann'}
Ressursbruk
	Kjøringen: 133ms
	Å konsumere: 135ms


I browseren:
```
MATCH (p:Person {MISTENKT=1})
return p
```


In [21]:
# Opprydding
# Fjerne projeksjonen
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('BakmannGraf', false) YIELD graphName"""
)
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person  {MISTENKT:1})
    REMOVE p.MISTENKT
    RETURN COUNT(p) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person)
    WHERE p.LCC > 0
    REMOVE p.LCC
    RETURN COUNT(p) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#


	Antall: 1
	Antall: 69


## Lete etter deling av utbytte

Har ikke funnet noen mæte å identifisere dette mønsteret på, uten å legge til grunn at jeg allerede vet svaret.